# SHAP Beeswarm with LASSO Coefficients

*Generated 2025-07-25*

This notebook reproduces the feature selection (LASSO) and explainability (SHAP beeswarm) steps described in the manuscript. It:

1. Loads a CSV of routine laboratory features and the binary target (NT-proBNP ≥300 pg/mL).
2. Splits the data into train/test (80/20).
3. Fits a LASSO (L1-penalized) logistic regression to select features.
4. Computes SHAP values with `shap.LinearExplainer` and plots a beeswarm.
5. Exports coefficients and SHAP summaries.

Adjust paths and column names as needed.

## 0. Environment
Make sure the following packages are installed:

```bash
pip install scikit-learn==1.6 shap==0.45 pandas numpy matplotlib seaborn
```

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, classification_report
import shap
import matplotlib.pyplot as plt
import seaborn as sns

print('Versions:')
import sklearn, shap as _shap
print('sklearn', sklearn.__version__)
print('shap', _shap.__version__)


## 1. Load data
Edit the path and target/feature columns as appropriate.

In [ ]:
# ==== USER SETTINGS ====
CSV_PATH = 'data/sample_data_with_target.csv'  # replace with your real file
TARGET_COL = 'NPB300'                          # binary target (1: NT-proBNP >=300 pg/mL)
ID_COL = None                                  # e.g., 'SQ' if you have an ID

# Columns to drop (NT-proBNP itself and 4 hematologic variables per manuscript)
DROP_COLS = ['NTproBNP','RBC','Hb','Hct','MCH']  # adjust names

# ==== LOAD ====
df = pd.read_csv(CSV_PATH)

if ID_COL and ID_COL in df.columns:
    df_id = df[ID_COL]
else:
    df_id = None

y = df[TARGET_COL].astype(int)
X = df.drop(columns=[c for c in [TARGET_COL] + DROP_COLS if c in df.columns])

print('Shape:', X.shape)
X.head()

## 2. Train / test split & scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=1, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print('Train:', X_train.shape, ' Test:', X_test.shape)

## 3. LASSO (L1) Logistic Regression

In [ ]:
# We use LogisticRegression with L1 penalty as a LASSO classifier
logit = LogisticRegression(penalty='l1', solver='liblinear', max_iter=2000, random_state=1, n_jobs=None)
logit.fit(X_train_s, y_train)

coef = pd.Series(logit.coef_[0], index=X.columns)
coef_abs = coef.abs().sort_values(ascending=False)

# Threshold for negligible coefficients
THRESH = 1e-7
selected_features = coef_abs[coef_abs >= THRESH].index.tolist()

print('Selected features (|coef| >= 1e-7):', selected_features)
display(coef_abs.to_frame('abs_coef').head(20))

## 4. Performance (for sanity check)

In [ ]:
proba_test = logit.predict_proba(X_test_s)[:,1]
pred_test = (proba_test >= 0.5).astype(int)

auc = roc_auc_score(y_test, proba_test)
print('AUROC:', auc)
print(classification_report(y_test, pred_test))


## 5. SHAP LinearExplainer & Beeswarm

In [ ]:
explainer = shap.LinearExplainer(logit, X_train_s, feature_names=X.columns)
shap_values = explainer.shap_values(X_train_s)

# Beeswarm plot
plt.figure(figsize=(8,6))
shap.summary_plot(shap_values, X_train, plot_type='dot', show=False)
plt.tight_layout()
plt.savefig('docs/shap_beeswarm.png', dpi=300)
plt.show()


## 6. Save outputs

In [ ]:
coef_df = pd.DataFrame({
    'feature': X.columns,
    'coef': logit.coef_[0],
    'abs_coef': np.abs(logit.coef_[0])
}).sort_values('abs_coef', ascending=False)

coef_df.to_csv('results/lasso_coefficients.csv', index=False)

np.save('results/shap_values.npy', shap_values)
X_train.to_csv('results/shap_X_train.csv', index=False)

print('Saved coefficients and SHAP arrays to results/ folder.')

## Notes
- Replace the synthetic CSV with your institution’s de-identified dataset.
- To match the manuscript exactly, ensure the same scaler, split seed, and LASSO settings.
- If you prefer DeLong CI for AUROC, use external helpers (not included here).